<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/time_honest_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final model comparison with time-based validation and confidence intervals

In `final_model_comparison.ipynb` we selected Random Forest Tuned, but every number came from one seed and a random validation split. Here we rerun the same models with three changes:

1. **Time-based validation.** Validation is the last 20% of the training period, so it comes after the data we train on, just like the test set.
2. **Five seeds.** We train each model five times (seeds 42–46) and report the average.
3. **Confidence intervals.** Every number gets a 95% interval from a bootstrap that resamples whole test days.

We also now fit `category_risk` on the training part only, so the validation rows are not used to build it.

Everything else stays the same: the six features, the test set (Kaggle's later `fraudTest.csv` period), the F2-optimal threshold chosen on validation, and the hyperparameters from `final_model_comparison.ipynb`.

**Why we resample whole days:** Sparkov creates fraud in bursts of 7–16 transactions on one day. Resampling whole days keeps each burst together, so our intervals are not too narrow.

## 1. Environment setup
The same setup as our earlier notebooks. We type credentials in at runtime, so they are never saved in the notebook.

In [1]:
# 1.1 Download both datasets from Kaggle
import os
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!kaggle datasets download -d mlg-ulb/creditcardfraud -p /content/data/ulb --unzip
!kaggle datasets download -d kartik2112/fraud-detection -p /content/data/sparkov --unzip

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:00<00:00, 149MB/s]

Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:01<00:00, 118MB/s]



In [2]:
# 1.2 Clone the project repository
github_token = getpass('Paste your GitHub token: ')
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 164 (delta 101), reused 17 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 11.52 MiB | 14.36 MiB/s, done.
Resolving deltas: 100% (101/101), done.


In [3]:
# 1.3 Move into the repository and place the data in data/
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
!mv /content/data/ulb /content/data/sparkov data/

/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [4]:
# 1.4 Libraries not preinstalled on Colab
!pip install -q catboost imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


### 1.5 Saving progress to Google Drive
Training all models takes a few hours. We save each finished model and seed to Google Drive straight away, so if Colab disconnects, re-running the notebook skips everything already finished.

In [6]:
# 1.5 Connect Google Drive for saving progress
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/fraud_checkpoints/time_honest_evaluation'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('checkpoints ->', CHECKPOINT_DIR)

Mounted at /content/drive
checkpoints -> /content/drive/MyDrive/fraud_checkpoints/time_honest_evaluation


In [7]:
# 1.6 Imports and project helpers
import sys
import numpy as np
import pandas as pd

sys.path.append(os.getcwd())
from src.split import split_sparkov
from src.utils import fit_category_risk, apply_category_risk, add_prev_txn_features
from src.evaluation import (chronological_val_split, run_seeds, bootstrap_model,
                            summarise, paired_difference, format_ci)

## 2. Six-feature Sparkov set
We build the features the same way as in `final_model_comparison.ipynb`. The only difference is that `category_risk` is built later, in Section 3, after the validation split.

Implied velocity is calculated on the full train + test timeline, so the first test transaction of each card can still look back at that card's last training transaction. It uses only earlier times and merchant locations, never fraud labels.

In [8]:
# 2.1 Kaggle's chronological train/test split, and implied velocity on the full timeline
sparkov_train, sparkov_test = split_sparkov()

sparkov_train['_split'] = 'train'
sparkov_test['_split']  = 'test'
full = pd.concat([sparkov_train, sparkov_test]).sort_values('trans_date_trans_time')
full = add_prev_txn_features(full, 'cc_num', 'trans_date_trans_time', 'merch_lat', 'merch_long')
sparkov_train = full[full['_split'] == 'train'].drop(columns='_split')
sparkov_test  = full[full['_split'] == 'test'].drop(columns='_split')
print(sparkov_train.shape, sparkov_test.shape)

Train: 1,296,675 rows, from 2019-01-01 00:00:18 to 2020-06-21 12:13:37
Test:  555,719 rows, from 2020-06-21 12:14:25 to 2020-12-31 23:59:34
(1296675, 26) (555719, 26)


In [9]:
# 2.2 Build the features that do not use fraud labels (same definitions as before)
def add_features(df):
    df = df.copy()
    ts = pd.to_datetime(df['trans_date_trans_time'])
    df['velocity_log'] = np.log1p(df['implied_velocity_kmh'].fillna(0))
    df['is_night']     = ts.dt.hour.isin([22, 23, 0, 1, 2, 3]).astype(int)
    df['amt_log']      = np.log1p(df['amt'])
    df['age']          = (ts - pd.to_datetime(df['dob'])).dt.days / 365.25
    df['gender_flag']  = (df['gender'] == 'M').astype(int)
    df['day']          = ts.dt.floor('D')   # used only by the bootstrap
    return df

sparkov_train = add_features(sparkov_train)
sparkov_test  = add_features(sparkov_test)

## 3. Time-based validation split
We set the cutoff 80% of the way through the training period. Transactions before the cutoff are used for training (and for fitting `category_risk`), and transactions after it form the validation set. The split function checks that no validation transaction is earlier than the training data.

In [10]:
# 3.1 Split by time, then fit category_risk on the training part only
TIME = 'trans_date_trans_time'
train_sub, val, cutoff = chronological_val_split(sparkov_train, TIME, val_frac=0.2)

# category_risk is learned from the training part and applied to validation and test
category_risk_map = fit_category_risk(train_sub, 'category', 'is_fraud')
for df in (train_sub, val, sparkov_test):
    df['category_risk'] = apply_category_risk(df, 'category', category_risk_map)

features = ['amt_log', 'category_risk', 'is_night', 'velocity_log', 'age', 'gender_flag']
target = 'is_fraud'
X_tr,   y_tr   = train_sub[features],    train_sub[target]
X_val,  y_val  = val[features],          val[target]
X_test, y_test = sparkov_test[features], sparkov_test[target]
test_days = sparkov_test['day'].values

print('cutoff:', cutoff)
for name, df in [('train_sub', train_sub), ('val', val), ('test', sparkov_test)]:
    t = pd.to_datetime(df[TIME])
    print(f"{name:9s} rows={len(df):>9,}  fraud rate={df[target].mean():.4%}  "
          f"{t.min():%Y-%m-%d} -> {t.max():%Y-%m-%d}  days={df['day'].nunique()}")

cutoff: 2020-03-06 00:10:57.200000
train_sub rows=1,036,901  fraud rate=0.5756%  2019-01-01 -> 2020-03-06  days=430
val       rows=  259,774  fraud rate=0.5921%  2020-03-06 -> 2020-06-21  days=108
test      rows=  555,719  fraud rate=0.3860%  2020-06-21 -> 2020-12-31  days=194


We print the dates and fraud rate of each part to confirm they are in time order. Because we no longer use a stratified split, the fraud rates of the training and validation parts are not forced to be equal.

## 4. Training each model with five seeds
We keep the hyperparameters chosen in `final_model_comparison.ipynb`. `scale_pos_weight` is recalculated from the new training part.

Random Forest Tuned and CatBoost Tuned take the longest. To get the key results sooner, `MODELS_TO_RUN` can be limited to these two plus XGBoost Weighted, and the other models added in a later session.

In [11]:
# 4.0 Models and their settings
from sklearn.ensemble import RandomForestClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

SEEDS = [42, 43, 44, 45, 46]
spw = (y_tr == 0).sum() / (y_tr == 1).sum()
print('scale_pos_weight:', round(spw, 2))

MODELS = {
    'Random Forest Tuned': dict(
        make=lambda s: RandomForestClassifier(n_estimators=200, max_features=3, min_samples_leaf=5,
                                              random_state=s, n_jobs=-1)),
    'CatBoost Tuned': dict(
        make=lambda s: CatBoostClassifier(iterations=2000, depth=8, learning_rate=0.1,
                                          eval_metric='PRAUC', early_stopping_rounds=100,
                                          random_seed=s, verbose=0, thread_count=-1),
        fit_kwargs=lambda s: {'eval_set': (X_val, y_val)}),
    'Random Forest Control': dict(
        make=lambda s: RandomForestClassifier(n_estimators=100, random_state=s, n_jobs=-1)),
    'CatBoost Control': dict(
        make=lambda s: CatBoostClassifier(iterations=500, random_seed=s, verbose=0, thread_count=-1)),
    'XGBoost Weighted': dict(
        make=lambda s: XGBClassifier(n_estimators=100, eval_metric='logloss',
                                     scale_pos_weight=spw, random_state=s)),
    'CatBoost Weighted': dict(
        make=lambda s: CatBoostClassifier(iterations=500, scale_pos_weight=spw,
                                          random_seed=s, verbose=0, thread_count=-1)),
    'Balanced RF': dict(
        make=lambda s: BalancedRandomForestClassifier(n_estimators=100, sampling_strategy='all',
                                                      replacement=True, bootstrap=False,
                                                      random_state=s, n_jobs=-1)),
}

MODELS_TO_RUN = list(MODELS)   # e.g. ['Random Forest Tuned', 'CatBoost Tuned', 'XGBoost Weighted']

scale_pos_weight: 172.74


In [12]:
# 4.1 Train every model with every seed (finished runs are loaded from Drive)
runs = {}
for name in MODELS_TO_RUN:
    spec = MODELS[name]
    runs[name] = run_seeds(name, spec['make'], X_tr, y_tr, X_val, y_val, X_test, SEEDS,
                           checkpoint_dir=CHECKPOINT_DIR, fit_kwargs=spec.get('fit_kwargs'))

Random Forest Tuned seed 42: threshold=0.15  val PR-AUC=0.8766
Random Forest Tuned seed 43: threshold=0.16  val PR-AUC=0.8772
Random Forest Tuned seed 44: threshold=0.15  val PR-AUC=0.8777
Random Forest Tuned seed 45: threshold=0.19  val PR-AUC=0.8759
Random Forest Tuned seed 46: threshold=0.14  val PR-AUC=0.8773
CatBoost Tuned seed 42: threshold=0.12  val PR-AUC=0.8650
CatBoost Tuned seed 43: threshold=0.14  val PR-AUC=0.8674
CatBoost Tuned seed 44: threshold=0.14  val PR-AUC=0.8651
CatBoost Tuned seed 45: threshold=0.11  val PR-AUC=0.8654
CatBoost Tuned seed 46: threshold=0.13  val PR-AUC=0.8684
Random Forest Control seed 42: threshold=0.16  val PR-AUC=0.8595
Random Forest Control seed 43: threshold=0.17  val PR-AUC=0.8589
Random Forest Control seed 44: threshold=0.15  val PR-AUC=0.8601
Random Forest Control seed 45: threshold=0.13  val PR-AUC=0.8577
Random Forest Control seed 46: threshold=0.21  val PR-AUC=0.8588
CatBoost Control seed 42: threshold=0.12  val PR-AUC=0.8473
CatBoost C

### 4.2 Optional: re-checking the Random Forest settings
Our settings (`max_features=3`, `min_samples_leaf=5`) were chosen on the old random validation split. This cell repeats the same six-setting search on the new validation set, using validation only. It takes about 30–40 minutes and is switched off by default.

In [13]:
# 4.2 Set to True to repeat the Random Forest settings search
RECHECK_RF = False

if RECHECK_RF:
    from sklearn.metrics import average_precision_score
    grid = [{'max_features': 'sqrt', 'min_samples_leaf': 1}, {'max_features': 3, 'min_samples_leaf': 1},
            {'max_features': 4, 'min_samples_leaf': 1},      {'max_features': 'sqrt', 'min_samples_leaf': 5},
            {'max_features': 3, 'min_samples_leaf': 5},      {'max_features': 3, 'min_samples_leaf': 10}]
    rows = []
    for g in grid:
        m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, **g).fit(X_tr, y_tr)
        rows.append({**g, 'Val PR-AUC': round(average_precision_score(y_val, m.predict_proba(X_val)[:, 1]), 4)})
        print(rows[-1])
    display(pd.DataFrame(rows).sort_values('Val PR-AUC', ascending=False))

## 5. Results with confidence intervals
### 5.1 Results table
For each model, we resample the test days 500 times. On each resample we score all five seeds and take their average. The table shows the average over seeds on the full test set, with the 95% interval in brackets. The seed SD shows how much the result changes between seeds.

All models use the same resampled days (`BOOT_SEED`), so we can compare them fairly in Section 5.2. This step takes about 6 minutes per model.

In [14]:
# 5.1 Bootstrap each model and build the results table
N_BOOT = 500
BOOT_SEED = 2026

boots, summaries = {}, {}
for name, seed_runs in runs.items():
    pairs = [(r['test_proba'], r['threshold']) for r in seed_runs]
    boots[name] = bootstrap_model(y_test, pairs, test_days, n_boot=N_BOOT, boot_seed=BOOT_SEED)
    summaries[name] = summarise(y_test, pairs, boots[name])
    summaries[name]['Val PR-AUC'] = np.mean([r['val_pr_auc'] for r in seed_runs])
    print(f"{name}: done")

summary_df = pd.DataFrame(summaries).T.sort_values('PR-AUC', ascending=False)

# results table: average [95% CI], plus seed SD for PR-AUC
table = pd.DataFrame({
    'Val PR-AUC': summary_df['Val PR-AUC'].astype(float).round(4),
    'PR-AUC [95% CI]': [format_ci(r, 'PR-AUC') for _, r in summary_df.iterrows()],
    'PR-AUC seed SD': summary_df['PR-AUC seed SD'].astype(float).round(4),
    'F1 [95% CI]':  [format_ci(r, 'F1')  for _, r in summary_df.iterrows()],
    'F2 [95% CI]':  [format_ci(r, 'F2')  for _, r in summary_df.iterrows()],
    'MCC [95% CI]': [format_ci(r, 'MCC') for _, r in summary_df.iterrows()],
    'Thresholds (per seed)': summary_df['Thresholds'],
})
table

Random Forest Tuned: done
CatBoost Tuned: done
Random Forest Control: done
CatBoost Control: done
XGBoost Weighted: done
CatBoost Weighted: done
Balanced RF: done


,Val PR-AUC,PR-AUC [95% CI],PR-AUC seed SD,F1 [95% CI],F2 [95% CI],MCC [95% CI],Thresholds (per seed)
Random Forest Tuned,0.8769,"0.8341 [0.8140, 0.8529]",0.0003,"0.7087 [0.6812, 0.7310]","0.7892 [0.7689, 0.8056]","0.7180 [0.6941, 0.7385]","0.15, 0.16, 0.15, 0.19, 0.14"
CatBoost Tuned,0.8663,"0.8192 [0.7968, 0.8385]",0.0019,"0.6720 [0.6451, 0.6957]","0.7733 [0.7533, 0.7912]","0.6873 [0.6631, 0.7088]","0.12, 0.14, 0.14, 0.11, 0.13"
Random Forest Control,0.8590,"0.8139 [0.7921, 0.8345]",0.0009,"0.7035 [0.6767, 0.7251]","0.7798 [0.7598, 0.7966]","0.7122 [0.6879, 0.7319]","0.16, 0.17, 0.15, 0.13, 0.21"
CatBoost Control,0.8500,"0.7981 [0.7721, 0.8191]",0.0017,"0.6589 [0.6317, 0.6834]","0.7557 [0.7345, 0.7737]","0.6731 [0.6492, 0.6948]","0.12, 0.12, 0.11, 0.11, 0.15"
XGBoost Weighted,0.8425,"0.7937 [0.7695, 0.8152]",0.0000,"0.6528 [0.6248, 0.6777]","0.7534 [0.7319, 0.7720]","0.6680 [0.6432, 0.6899]","0.92, 0.92, 0.92, 0.92, 0.92"
CatBoost Weighted,0.8341,"0.7747 [0.7473, 0.7997]",0.0055,"0.6313 [0.6025, 0.6581]","0.7411 [0.7198, 0.7604]","0.6503 [0.6251, 0.6729]","0.93, 0.93, 0.84, 0.90, 0.85"
Balanced RF,0.7843,"0.7269 [0.6971, 0.7529]",0.0056,"0.6572 [0.6280, 0.6828]","0.7600 [0.7384, 0.7772]","0.6730 [0.6474, 0.6946]","0.88, 0.88, 0.88, 0.89, 0.88"


### 5.2 Is Random Forest Tuned really better? Paired differences
For each other model, we calculate Random Forest Tuned's score minus that model's score on every resample. If the 95% interval does not include zero, we treat the difference as real. `P(A > B)` is how often Random Forest Tuned scored higher.

In [15]:
# 5.2 Paired differences against Random Forest Tuned
REF = 'Random Forest Tuned'
diff_rows = []
for name in boots:
    if name == REF:
        continue
    for metric in ['PR-AUC', 'F2', 'MCC']:
        d = paired_difference(boots[REF], boots[name], metric)
        diff_rows.append({'vs': name, 'Metric': metric, **{k: round(v, 4) for k, v in d.items()}})
diff_df = pd.DataFrame(diff_rows)
diff_df

,vs,Metric,mean diff,CI low,CI high,P(A > B)
0,CatBoost Tuned,PR-AUC,0.0148,0.0085,0.0222,1.000
1,CatBoost Tuned,F2,0.0157,0.0085,0.0227,1.000
2,CatBoost Tuned,MCC,0.0305,0.0222,0.0381,1.000
3,Random Forest Control,PR-AUC,0.0203,0.0167,0.0239,1.000
4,Random Forest Control,F2,0.0094,0.0049,0.0134,1.000
5,Random Forest Control,MCC,0.0058,0.0015,0.0099,0.994
6,CatBoost Control,PR-AUC,0.0359,0.0275,0.0455,1.000
7,CatBoost Control,F2,0.0335,0.0238,0.0434,1.000
8,CatBoost Control,MCC,0.0449,0.0355,0.0541,1.000
9,XGBoost Weighted,PR-AUC,0.0401,0.0313,0.0505,1.000


In [16]:
# 5.3 Save the results tables
os.makedirs('results', exist_ok=True)
summary_df.to_csv('results/time_honest_summary.csv')
table.to_csv('results/time_honest_table.csv')
diff_df.to_csv('results/time_honest_paired_differences.csv', index=False)
pd.DataFrame([{'model': n, 'seed': r['seed'], 'threshold': r['threshold'], 'val_pr_auc': r['val_pr_auc']}
              for n, rs in runs.items() for r in rs]).to_csv('results/time_honest_per_seed.csv', index=False)
print('saved to results/')

saved to results/


## 6. Discussion
*(We complete this section after the run, comparing the results with `final_model_comparison.ipynb`.)*

We look at five points:
- whether the gap between validation and test PR-AUC is smaller than before (0.8837 vs 0.8326 for Random Forest Tuned);
- whether the F2-optimal thresholds have changed from the old ones (0.14 for Random Forest Tuned);
- whether Random Forest Tuned is still clearly ahead of CatBoost Tuned, or whether the two are tied;
- whether the seed SD is small compared with the interval width;
- whether the smaller training part (the first 80% of the training period) lowers the test scores.

**Limitations:**
- The hyperparameters were chosen on the earlier random validation split, unless Section 4.2 confirms them.
- The intervals reflect which days appear in the test period. They do not cover drift after December 2020.
- Resampling days keeps each fraud burst together, but not each card's full history.